In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os, glob
from pathlib import Path

In [2]:
DATA_DIR = Path("../data/raw")
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR  = DATA_DIR / "test"

_train_files = sorted(glob.glob(os.path.join(TRAIN_DIR, '*__horizontal_well.csv')))
TRAIN_WIDS = [os.path.basename(f).split('__')[0] for f in _train_files]
_test_files = sorted(glob.glob(os.path.join(TEST_DIR, '*__horizontal_well.csv')))
TEST_WIDS = [os.path.basename(f).split('__')[0] for f in _test_files]

print(f"훈련 데이터 총 우물 수: {len(TRAIN_WIDS)}")
print(f"훈련 데이터 우물 ID: {TRAIN_WIDS[:5]}")

def load_well(wid, split='train'):
    base = TRAIN_DIR if split == 'train' else TEST_DIR
    hw = pd.read_csv(os.path.join(base, f'{wid}__horizontal_well.csv'))
    tw = pd.read_csv(os.path.join(base, f'{wid}__typewell.csv'))
    return hw, tw

훈련 데이터 총 우물 수: 773
훈련 데이터 우물 ID: ['000d7d20', '00bbac68', '00e12e8b', '015fe0d2', '01869cd4']


In [7]:
import xgboost as xgb
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import LabelEncoder

markers = ['ANCC','ASTNU','ASTNL','EGFDU','EGFDL','BUDA']
Xs, ys, gs = [], [], []
for wid in TRAIN_WIDS[:150]:
    hw, _ = load_well(wid, 'train')
    z = hw['Z'].values.astype(float)
    layer = (z[:,None] > hw[markers].values).sum(1)        # 지층 라벨(0~6)
    gr = hw['GR'].interpolate(limit_direction='both').fillna(hw['GR'].mean()).values
    feat = pd.DataFrame({
        'gr': gr, 'z': z, 'grad': np.gradient(gr, hw['MD'].values),
        'gr_m15': pd.Series(gr).rolling(15,center=True,min_periods=1).mean(),
        'gr_s15': pd.Series(gr).rolling(15,center=True,min_periods=1).std().fillna(0),
        'x': hw['X'].values, 'y': hw['Y'].values,
    })
    Xs.append(feat); ys.append(layer); gs.append(np.full(len(hw), wid))
X = pd.concat(Xs, ignore_index=True); y = np.concatenate(ys)
y = LabelEncoder().fit_transform(np.concatenate(ys))
g = np.concatenate(gs)

oof = np.zeros(len(y))
for tr, va in GroupKFold(5).split(X, y, g):
    m = xgb.XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05,
                          subsample=0.8, tree_method='hist')
    m.fit(X.iloc[tr], y[tr])
    oof[va] = m.predict(X.iloc[va])
print('지층 분류 정확도:', (oof == y).mean())


지층 분류 정확도: 0.8558482070942192


In [3]:
import xgboost as xgb
from sklearn.model_selection import GroupKFold

markers = ['ANCC','ASTNU','ASTNL','EGFDU','EGFDL','BUDA']
Xs, ys, gs = [], [], []
for wid in TRAIN_WIDS[:150]:
    hw, _ = load_well(wid, 'train')
    z = hw['Z'].values.astype(float)
    layer = (z[:, None] > hw[markers].values).sum(1)        # 정답 지층(oracle)
    feat = pd.DataFrame({'layer': layer, 'z': z,
                         'x': hw['X'].values, 'y': hw['Y'].values})
    Xs.append(feat); ys.append(hw['TVT'].values.astype(float))
    gs.append(np.full(len(hw), wid))
X = pd.concat(Xs, ignore_index=True); y = np.concatenate(ys); g = np.concatenate(gs)

oof = np.zeros(len(y))
for tr, va in GroupKFold(5).split(X, y, g):
    m = xgb.XGBRegressor(n_estimators=400, max_depth=6, learning_rate=0.05,
                         subsample=0.8, tree_method='hist')
    m.fit(X.iloc[tr], y[tr]); oof[va] = m.predict(X.iloc[va])
print('oracle 지층+위치 → TVT RMSE:', np.sqrt(((oof - y) ** 2).mean()))


oracle 지층+위치 → TVT RMSE: 169.80821624856537


In [4]:
markers = ['ANCC','ASTNU','ASTNL','EGFDU','EGFDL','BUDA']
Xs, ys, gs = [], [], []
for wid in TRAIN_WIDS[:150]:
    hw, _ = load_well(wid, 'train')
    z = hw['Z'].values.astype(float)
    layer = (z[:, None] > hw[markers].values).sum(1)
    kn = hw['TVT_input'].notna().values
    base = np.median((hw['TVT_input'].values + z)[kn])      # ★ known C 레벨(절대 anchor)
    feat = pd.DataFrame({'layer': layer, 'z': z,
                         'x': hw['X'].values, 'y': hw['Y'].values,
                         'base': base})                      # 우물별 절대 레벨
    Xs.append(feat); ys.append(hw['TVT'].values.astype(float))
    gs.append(np.full(len(hw), wid))
X = pd.concat(Xs, ignore_index=True); y = np.concatenate(ys); g = np.concatenate(gs)

oof = np.zeros(len(y))
for tr, va in GroupKFold(5).split(X, y, g):
    m = xgb.XGBRegressor(n_estimators=400, max_depth=6, learning_rate=0.05,
                         subsample=0.8, tree_method='hist')
    m.fit(X.iloc[tr], y[tr]); oof[va] = m.predict(X.iloc[va])
print('oracle 지층 + known anchor → TVT RMSE:', np.sqrt(((oof - y) ** 2).mean()))

oracle 지층 + known anchor → TVT RMSE: 118.67604716728893
